# Modeling

## Imports

In [1]:
import copy
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from utils.experiment_separator import (
    get_experiments,
    print_experiment_report,
    separate_experiments,
)

## Constants and Hyperparameters

In [2]:
SEED = 42
SEEDS = [42, 123, 456, 789, 2026]
RUN_SINGLE_SEED = False
DATA_PATH = "../data/data.csv"
COLUMN_NAMES = ["density", "cutting_speed", "feed_rate", "depth", "axial_force", "cutting_force"]
TARGET = "cutting_force"
INPUT = ["density", "cutting_speed", "feed_rate", "depth", "axial_force"]
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
WINDOW_SIZE = 100
STRIDE = 5
HIDDEN_SIZE = 128
NUM_LAYERS = 2
EPOCHS = 10
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
PATIENCE = 2
MIN_DELTA = 0.0
TORCH_NUM_THREADS = 8
TORCH_NUM_INTEROP_THREADS = 2
INPUT_SIZE = len(INPUT)
OUTPUT_SIZE = 1

## Definitions

### `configure_runtime`

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def configure_runtime():
    if not getattr(configure_runtime, "configured", False):
        torch.set_num_threads(TORCH_NUM_THREADS)
        torch.set_num_interop_threads(TORCH_NUM_INTEROP_THREADS)
        torch.backends.mkldnn.enabled = True
        configure_runtime.configured = True

### `apply_scaling`

In [4]:
def apply_scaling(df: pd.DataFrame, scaler: StandardScaler, fit: bool = False) -> pd.DataFrame:
    cols = INPUT + [TARGET]
    if fit:
        df[cols] = scaler.fit_transform(df[cols])
    else:
        df[cols] = scaler.transform(df[cols])
    return df

### `Pipeline`

In [5]:
class Pipeline:
    def __init__(self):
        self.scaler = StandardScaler()
 
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_scaling(temp, self.scaler, fit=True)
        return temp
 
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_scaling(temp, self.scaler, fit=False)
        return temp

### `split_experiment_keys`

In [6]:
def split_experiment_keys(experiments, seed):
    keys = list(experiments.keys())
    random.Random(seed).shuffle(keys)
    train_end = int(len(keys) * TRAIN_RATIO)
    validation_end = train_end + int(len(keys) * VALIDATION_RATIO)
    return keys[:train_end], keys[train_end:validation_end], keys[validation_end:]

### `experiments_to_tensor`

In [7]:
def experiments_to_tensor(keys, experiments, pipeline, window_size, fit=False):
    ref_df = pd.concat([experiments[k] for k in keys], ignore_index=True)
    processed = pipeline.fit_transform(ref_df) if fit else pipeline.transform(ref_df)
 
    lengths = [len(experiments[k]) for k in keys]
 
    X_windows, y_windows = [], []
    start = 0
    for length in lengths:
        exp_slice = processed.iloc[start:start+length].reset_index(drop=True)
        start += length
 
        if len(exp_slice) < window_size:
            continue
 
        input_vals  = exp_slice[INPUT].values
        target_vals = exp_slice[TARGET].values
 
        for i in range(0, len(exp_slice) - window_size + 1, STRIDE):
            X_windows.append(input_vals[i:i+window_size])
            y_windows.append(target_vals[i+window_size-1])
 
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
    return X, y

### `calculate_loss_in_batches`

In [8]:
def calculate_loss_in_batches(model, X, y, criterion, batch_size):
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=False)
    total_loss = 0.0
    total_samples = 0
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            loss = criterion(model(X_batch), y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_samples += X_batch.size(0)
    return total_loss / total_samples

### `prepare_datasets and train`

In [9]:
def prepare_datasets(experiments, train_keys, val_keys, test_keys):
    """Create scaled windows once; LSTM, GRU, and RNN reuse them."""
    pipeline = Pipeline()
    X_train, y_train = experiments_to_tensor(train_keys, experiments, pipeline, WINDOW_SIZE, fit=True)
    X_val, y_val     = experiments_to_tensor(val_keys,   experiments, pipeline, WINDOW_SIZE, fit=False)
    X_test, y_test   = experiments_to_tensor(test_keys,  experiments, pipeline, WINDOW_SIZE, fit=False)
    print(f"Prepared tensors | train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,} | stride: {STRIDE}")
    return {
        'train_dataset': TensorDataset(X_train, y_train),
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'pipeline': pipeline,
    }

def train(model_class, model_name: str, prepared_data: dict, seed: int):
    set_seed(seed)
    print(f"\nTraining {model_name} ")
    model = model_class()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=model.lr)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(prepared_data['train_dataset'], batch_size=model.batch_size, shuffle=True, generator=generator)

    best_val_loss = float('inf')
    best_state_dict = None
    best_epoch = -1
    epochs_without_improvement = 0

    for epoch in range(model.epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        val_loss = calculate_loss_in_batches(
            model, prepared_data['X_val'], prepared_data['y_val'], criterion, model.batch_size
        )
        train_loss /= len(train_loader)
        print(f"  Epoch {epoch + 1}/{model.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"  Early stopping at epoch {epoch + 1} (patience={PATIENCE})")
                break

    print(f"  Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")
    model.load_state_dict(best_state_dict)
    model.best_epoch = best_epoch
    model.best_val_loss = best_val_loss
    return model

### `inverse_transform_target`

In [10]:
def inverse_transform_target(pipeline, values):
    n_cols = len(INPUT) + 1
    dummy = np.zeros((len(values), n_cols))
    dummy[:, -1] = values
    return pipeline.scaler.inverse_transform(dummy)[:, -1]

### `evaluate`

In [11]:
def evaluate(model, model_name, X_test, y_test, pipeline):
    loader = DataLoader(TensorDataset(X_test, y_test), batch_size=model.batch_size, shuffle=False)
    model.eval()
    with torch.no_grad():
        predictions = torch.cat([model(X_batch).cpu() for X_batch, _ in loader]).squeeze().numpy()
    actuals = y_test.squeeze().numpy()
    predictions = inverse_transform_target(pipeline, predictions)
    actuals = inverse_transform_target(pipeline, actuals)
    mse = mean_squared_error(actuals, predictions)
    return {"Model": model_name, "Best Epoch": model.best_epoch, "Validation Loss": model.best_val_loss, "MAE": mean_absolute_error(actuals, predictions), "MSE": mse, "RMSE": np.sqrt(mse)}

### `compute_experiment_predictions`

In [12]:
def compute_experiment_predictions(model, key, experiments, pipeline):
    exp_df = experiments[key].copy()
    processed = pipeline.transform(exp_df)
 
    if len(processed) < WINDOW_SIZE:
        return None
    input_vals  = processed[INPUT].values
    target_vals = processed[TARGET].values
    depth_vals  = exp_df['depth'].values
    X_windows, y_windows, depth_windows = [], [], []
    for i in range(0, len(processed) - WINDOW_SIZE + 1, STRIDE):
        X_windows.append(input_vals[i:i+WINDOW_SIZE])
        y_windows.append(target_vals[i+WINDOW_SIZE-1])
        depth_windows.append(depth_vals[i+WINDOW_SIZE-1])
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
 
    model.eval()
    with torch.no_grad():
        preds = model(X).squeeze().numpy()
    actuals = y.squeeze().numpy()
 
    preds_orig   = inverse_transform_target(pipeline, preds)
    actuals_orig = inverse_transform_target(pipeline, actuals)
    depth_orig   = np.array(depth_windows)
 
    return depth_orig, actuals_orig, preds_orig

### `plot_validation_overview`

In [13]:
def plot_validation_overview(model, model_name, eval_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    exp_numbers, mean_actuals, mean_preds = [], [], []
    all_actuals, all_preds = [], []
 
    for idx, key in enumerate(eval_keys, start=1):
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        _, actuals_orig, preds_orig = result
        exp_numbers.append(idx)
        mean_actuals.append(np.mean(actuals_orig))
        mean_preds.append(np.mean(preds_orig))
        all_actuals.extend(actuals_orig)
        all_preds.extend(preds_orig)
 
    rmse_of_means   = np.sqrt(mean_squared_error(mean_actuals, mean_preds))
    rmse_per_window = np.sqrt(mean_squared_error(all_actuals, all_preds))
 
    plt.figure(figsize=(10, 4))
    plt.plot(exp_numbers, mean_actuals, label="Actual",    color="steelblue", marker='o')
    plt.plot(exp_numbers, mean_preds,   label="Predicted", color="tomato", linestyle="--", marker='x')
    plt.title(f"{model_name} \u2014 Test Overview (per-window RMSE: {rmse_per_window:.4f})")
    plt.xlabel("Experiment Number")
    plt.ylabel("cutting_force (mean, unscaled)")
    plt.xticks(exp_numbers)
    plt.legend()
    plt.tight_layout()
 
    save_path = os.path.join(out_dir, f"{model_name}_test_overview.png")
    plt.savefig(save_path)
    plt.close()
    print(f"  Saved: {save_path}  |  RMSE of per-exp means: {rmse_of_means:.4f}  |  True per-window RMSE: {rmse_per_window:.4f}")

### `plot_and_save_per_experiment`

In [14]:
def plot_and_save_per_experiment(model, model_name, eval_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    for key in eval_keys:
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        depth_orig, actuals_orig, preds_orig = result
 
        sort_idx       = np.argsort(depth_orig)
        depth_sorted   = depth_orig[sort_idx]
        actuals_sorted = actuals_orig[sort_idx]
        preds_sorted   = preds_orig[sort_idx]
 
        rmse_exp = np.sqrt(mean_squared_error(actuals_orig, preds_orig))
 
        exp_name  = "_".join(str(k) for k in key) if isinstance(key, tuple) else str(key)
        safe_name = exp_name.replace(" ", "").replace("/", "-")
 
        plt.figure(figsize=(10, 4))
        plt.plot(depth_sorted, actuals_sorted, label="Actual", color="steelblue", marker='o')
        plt.plot(depth_sorted, preds_sorted,   label="Predicted", color="tomato", linestyle="--", marker='x')
        plt.title(f"{model_name} RMSE: {rmse_exp:.4f} (Experiment {exp_name})")
        plt.xlabel("depth")
        plt.ylabel("cutting_force")
        plt.legend()
        plt.tight_layout()
 
        save_path = os.path.join(out_dir, f"{safe_name}.png")
        plt.savefig(save_path)
        plt.close()
        print(f"  Saved: {save_path}  |  RMSE: {rmse_exp:.4f}")

### LSTM

In [15]:
class LSTM(nn.Module):
    hidden_size = HIDDEN_SIZE
    num_layers  = NUM_LAYERS
    epochs      = EPOCHS
    batch_size  = BATCH_SIZE
    lr          = LEARNING_RATE
 
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc   = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

### `GRU`

In [16]:
class GRU(nn.Module):
    hidden_size = HIDDEN_SIZE
    num_layers  = NUM_LAYERS
    epochs      = EPOCHS
    batch_size  = BATCH_SIZE
    lr          = LEARNING_RATE
 
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc  = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        _, h_n = self.gru(x)
        return self.fc(h_n[-1])

### `RNN`

In [17]:
class RNN(nn.Module):
    hidden_size = HIDDEN_SIZE
    num_layers  = NUM_LAYERS
    epochs      = EPOCHS
    batch_size  = BATCH_SIZE
    lr          = LEARNING_RATE
 
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc  = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.fc(h_n[-1])

## Data Preparation

In [18]:
configure_runtime()
df = pd.read_csv(DATA_PATH)
df.columns = COLUMN_NAMES
df = separate_experiments(df)
print_experiment_report(df)
experiments = get_experiments(df)
if RUN_SINGLE_SEED:
    train_keys, val_keys, test_keys = split_experiment_keys(experiments, SEED)
    prepared_data = prepare_datasets(experiments, train_keys, val_keys, test_keys)

Total unique combinations : 72
Experiment count range    : 2 – 3
Total experiments         : 198

── Experiments per combination ──
 density  cutting_speed  feed_rate  experiment_count
      10             10         10                 3
      10             10         15                 3
      10             10         20                 3
      10             16         10                 3
      10             16         15                 3
      10             16         20                 3
      10             25         10                 3
      10             25         15                 3
      10             25         20                 3
      10             40         10                 3
      10             40         15                 3
      10             40         20                 3
      10             63         10                 3
      10             63         15                 3
      10             63         20                 3
      10            

## LSTM

In [19]:
if RUN_SINGLE_SEED:
    lstm_model = train(LSTM, "LSTM", prepared_data, SEED)
    lstm_performance = evaluate(lstm_model, "LSTM", prepared_data["X_test"], prepared_data["y_test"], prepared_data["pipeline"])

# plot_validation_overview(lstm_model, "LSTM", test_keys, experiments, prepared_data["pipeline"])
# plot_and_save_per_experiment(lstm_model, "LSTM", test_keys, experiments, prepared_data["pipeline"])

## GRU

In [20]:
if RUN_SINGLE_SEED:
    gru_model = train(GRU, "GRU", prepared_data, SEED)
    gru_performance = evaluate(gru_model, "GRU", prepared_data["X_test"], prepared_data["y_test"], prepared_data["pipeline"])

# plot_validation_overview(gru_model, "GRU", test_keys, experiments, prepared_data["pipeline"])
# plot_and_save_per_experiment(gru_model, "GRU", test_keys, experiments, prepared_data["pipeline"])

## RNN

In [21]:
if RUN_SINGLE_SEED:
    rnn_model = train(RNN, "RNN", prepared_data, SEED)
    rnn_performance = evaluate(rnn_model, "RNN", prepared_data["X_test"], prepared_data["y_test"], prepared_data["pipeline"])

# plot_validation_overview(rnn_model, "RNN", test_keys, experiments, prepared_data["pipeline"])
# plot_and_save_per_experiment(rnn_model, "RNN", test_keys, experiments, prepared_data["pipeline"])

## Performance Summary

In [22]:
if RUN_SINGLE_SEED:
    performance_summary = pd.DataFrame([lstm_performance, gru_performance, rnn_performance]).set_index("Model")
    print(performance_summary.round(4))

## Multi-Seed Evaluation

In [23]:
def run_multiseed(experiments, seeds):
    model_specs = [(LSTM, "LSTM"), (GRU, "GRU"), (RNN, "RNN")]
    results = []

    for seed in seeds:
        print(f"\n{'=' * 20} Seed {seed} {'=' * 20}")
        train_keys, val_keys, test_keys = split_experiment_keys(experiments, seed)
        prepared_data = prepare_datasets(experiments, train_keys, val_keys, test_keys)

        for model_class, model_name in model_specs:
            model = train(model_class, model_name, prepared_data, seed)
            performance = evaluate(
                model,
                model_name,
                prepared_data["X_test"],
                prepared_data["y_test"],
                prepared_data["pipeline"],
            )
            performance["Seed"] = seed
            results.append(performance)

    return pd.DataFrame(results)


def summarize_multiseed(results):
    metrics = ["Validation Loss", "MAE", "MSE", "RMSE"]
    rows = []

    for model_name, group in results.groupby("Model", sort=False):
        row = {"Model": model_name}
        row["Best Epoch"] = f"{group['Best Epoch'].mean():.2f} ± {group['Best Epoch'].std(ddof=1):.2f}"
        for metric in metrics:
            row[metric] = f"{group[metric].mean():.4f} ± {group[metric].std(ddof=1):.4f}"
        rows.append(row)

    return pd.DataFrame(rows).set_index("Model")


multiseed_results = run_multiseed(experiments, SEEDS)
multiseed_summary = summarize_multiseed(multiseed_results)
display(multiseed_results.round(4))
display(multiseed_summary)


==================== Seed 42 ====================
Prepared tensors | train: 37,836 | val: 7,435 | test: 7,595 | stride: 5

Training LSTM 
  Epoch 1/10 | Train Loss: 0.0445 | Val Loss: 0.0057
  Epoch 2/10 | Train Loss: 0.0032 | Val Loss: 0.0026
  Epoch 3/10 | Train Loss: 0.0020 | Val Loss: 0.0057
  Epoch 4/10 | Train Loss: 0.0016 | Val Loss: 0.0063
  Early stopping at epoch 4 (patience=2)
  Best epoch: 2 | Best Val Loss: 0.0026

Training GRU 
  Epoch 1/10 | Train Loss: 0.0476 | Val Loss: 0.0050
  Epoch 2/10 | Train Loss: 0.0046 | Val Loss: 0.0058
  Epoch 3/10 | Train Loss: 0.0029 | Val Loss: 0.0065
  Early stopping at epoch 3 (patience=2)
  Best epoch: 1 | Best Val Loss: 0.0050

Training RNN 
  Epoch 1/10 | Train Loss: 0.0425 | Val Loss: 0.0081
  Epoch 2/10 | Train Loss: 0.0054 | Val Loss: 0.0064
  Epoch 3/10 | Train Loss: 0.0036 | Val Loss: 0.0043
  Epoch 4/10 | Train Loss: 0.0024 | Val Loss: 0.0048
  Epoch 5/10 | Train Loss: 0.0025 | Val Loss: 0.0021
  Epoch 6/10 | Train Loss: 0.0023

,Model,Best Epoch,Validation Loss,MAE,MSE,RMSE,Seed
0,LSTM,2,0.0026,11.4679,385.2688,19.6283,42
1,GRU,1,0.0050,15.5091,690.8101,26.2833,42
2,RNN,5,0.0021,9.7035,321.7154,17.9364,42
3,LSTM,4,0.0021,5.8313,61.3542,7.8329,123
4,GRU,4,0.0023,6.8068,76.9764,8.7736,123
5,RNN,7,0.0017,6.2438,73.3767,8.5660,123
6,LSTM,4,0.0025,8.6994,192.0449,13.8580,456
7,GRU,4,0.0025,9.4684,236.4828,15.3780,456
8,RNN,9,0.0020,8.4101,157.9077,12.5661,456
9,LSTM,3,0.0032,8.0186,139.9640,11.8306,789


,Best Epoch,Validation Loss,MAE,MSE,RMSE
Model,,,,,
LSTM,3.40 ± 0.89,0.0033 ± 0.0015,8.0549 ± 2.2494,172.1309 ± 129.6631,12.4413 ± 4.6563
GRU,3.60 ± 1.82,0.0039 ± 0.0015,9.0938 ± 3.8181,242.7729 ± 259.3387,14.1166 ± 7.3734
RNN,6.20 ± 1.92,0.0034 ± 0.0020,8.1247 ± 1.6588,170.9444 ± 101.4825,12.6128 ± 3.8508
